# Train — 산술 전문가 3종(LGBM·XGBoost·CatBoost) 학습 + 로짓 내보내기

**카탈로그 기반**: [feature_catalog.csv](feature_catalog.csv) 의 사용=1 피쳐만 불러온다.
유형 플래그(아래 FLAGS)는 기본 전부 on — 끄는 실험은 Ablation.ipynb 의 몫이다.

**데이터 규율** (npz 미사용):

| 집합 | 행 | 역할 |
|---|---|---|
| `origin/train.csv` | 46,729 | 그래디언트 (5-fold, `arith_folds.csv`) |
| `origin/valid.csv` | 5,000 | early stopping 전용 + 로짓 내보내기 (2차 앙상블 재료) |
| `origin/holdout.csv` | 5,748 | **평가 전용** — 학습·선택에 절대 노출 금지 |

**swap 증강 + TTA**: 학습은 원본+swap 쌍둥이 2배(라벨 a↔b 교환), 예측은 swap-TTA 평균
(V4 test4 실측 −0.0040). swap 매핑은 카탈로그의 swap 컬럼에서 구성한다.

**부스터 3종** (모두 같은 fold·같은 ES·같은 swap 증강 — 향후 앙상블 재료):

| 모델 | 주요 파라미터 | 출처 |
|---|---|---|
| LGBM | lr 0.03, n 3000, leaves 63, min_child 100, ES 100 | V4 test4 채택값 |
| CatBoost | lr 0.03, iter 4000, depth 8, l2 4, Bernoulli 0.8, ES 200 | V4 test2 (ft 아키텍처) |
| XGBoost | lr 0.03, n 4000, depth 8, min_child 8, hist, ES 200 | 위 둘에 준한 구성 |

**산출물**: `model/{lgbm,xgb,cat}_fold1..5.*` + `model/meta.json` +
`outputs/arith_experts.npz` (train OOF / valid / holdout 의 log p — 2차 부스터 재료) +
`model/arith_importance.csv`

**선행**: Features.ipynb 완료. **예상 시간**: **~20–40분** (LGBM ~6분 / XGB ~8분 / CatBoost ~15분).

In [ ]:
# ── 설정 + 로드 (카탈로그 기반) ──
import os, sys, json, time
import numpy as np
import pandas as pd
from tqdm import tqdm

SEED = 42
META_COLS = ['id', 'role', 'label', 'is_tie']
# ── 데이터 로드 — origin/ 3분할. 훈련=train, ES=valid, 평가=holdout (npz 미사용) ──
def labels_of(d):
    yy = np.select([d['winner_model_a'] == 1, d['winner_model_b'] == 1,
                    d['winner_tie'] == 1], [0, 1, 2], default=-1).astype(np.int64)
    assert (yy != -1).all(), 'invalid labels!'
    return yy

parts = {}
for name, n_expect in [('train', 46729), ('valid', 5000), ('holdout', 5748)]:
    d = pd.read_csv(os.path.join('origin', f'{name}.csv'))
    assert len(d) == n_expect, f'{name}.csv {len(d)}행 != {n_expect}'
    d['role'] = name
    parts[name] = d
df = pd.concat([parts['train'], parts['valid'], parts['holdout']], ignore_index=True)
assert df['id'].is_unique, 'id 중복!'
y = labels_of(df)
role = df['role'].to_numpy()
m_tr, m_va, m_ho = role == 'train', role == 'valid', role == 'holdout'
y_tr, y_va, y_ho = y[m_tr], y[m_va], y[m_ho]
print(f'train {m_tr.sum():,} / valid {m_va.sum():,} / holdout {m_ho.sum():,}')
print('라벨 분포(a/b/tie):', np.round(np.bincount(y, minlength=3) / len(y), 4))

feat = pd.read_parquet('features_57k.parquet')
assert (feat['id'].to_numpy() == df['id'].to_numpy()).all(), 'features_57k 정렬 불일치!'
assert (feat['label'].to_numpy() == y).all(), '라벨 불일치!'
F = feat.drop(columns=META_COLS)

catalog = pd.read_csv('feature_catalog.csv').fillna({'제외사유': ''})
assert set(catalog['feature']) == set(F.columns), '카탈로그·파케이 불일치 — Features 재실행'

folds_df = pd.read_csv('arith_folds.csv')
assert (folds_df['id'].to_numpy() == df.loc[m_tr, 'id'].to_numpy()).all(), 'folds 불일치!'
folds = folds_df['fold'].to_numpy()

# 유형 플래그 — 기본 전부 on. 끄는 실험은 Ablation 에서.
FLAGS = {g: True for g in catalog['유형'].unique()}
cat_use = catalog[(catalog['사용'] == 1) & catalog['유형'].map(FLAGS)]
UCOLS = cat_use['feature'].tolist()
print(f"카탈로그 {len(catalog)}종 -> 사용 {int(catalog['사용'].sum())} -> 플래그 적용 {len(UCOLS)}종")
print(cat_use['유형'].value_counts().to_string())

X_tr = F.loc[m_tr, UCOLS].to_numpy(np.float32)
X_va = F.loc[m_va, UCOLS].to_numpy(np.float32)     # ES + 로짓 내보내기
X_ho = F.loc[m_ho, UCOLS].to_numpy(np.float32)     # 평가 전용

In [ ]:
# ── swap 매핑 — 카탈로그의 swap 컬럼이 진실의 원천 (A↔B 교환 시 피쳐 거동) ──
def build_swap(cols, swap_series):
    neg = [c for c in cols if swap_series[c] == '부호반전']
    recip = [c for c in cols if swap_series[c] == '역수']
    pairs = []
    for c in cols:
        v = swap_series[c]
        if isinstance(v, str) and v.startswith('교환:') and c < v.split(':', 1)[1]:
            partner = v.split(':', 1)[1]
            assert partner in set(cols), f'교환 파트너 {partner} 미포함 — 유형 플래그가 쌍을 갈랐다'
            pairs.append((c, partner))
    inv = [c for c in cols if swap_series[c] == '불변']
    assert len(neg) + len(recip) + len(inv) + 2 * len(pairs) == len(cols), 'swap 분류 누락!'
    idx = {c: i for i, c in enumerate(cols)}
    neg_i = np.array([idx[c] for c in neg], int)
    rec_i = np.array([idx[c] for c in recip], int)
    a_i = np.array([idx[a] for a, _ in pairs], int)
    b_i = np.array([idx[b] for _, b in pairs], int)

    def swap_X(X):
        Xs = X.copy()
        if len(neg_i):
            Xs[:, neg_i] *= -1.0
        if len(rec_i):
            Xs[:, rec_i] = 1.0 / np.clip(Xs[:, rec_i], 1e-12, None)
        if len(a_i):
            Xs[:, a_i], Xs[:, b_i] = Xs[:, b_i].copy(), Xs[:, a_i].copy()
        return Xs
    info = {'pairs': pairs, 'neg': neg, 'recip': recip, 'inv': inv}
    return swap_X, info

def swap_p3(p):
    return p[:, [1, 0, 2]]

def swap_label(yy):
    return np.where(yy == 0, 1, np.where(yy == 1, 0, 2)).astype(yy.dtype)

sw_series = catalog.set_index('feature')['swap']
swap_X, sw_info = build_swap(UCOLS, sw_series)
print(f"swap: 교환 {len(sw_info['pairs'])}쌍 / 부호반전 {len(sw_info['neg'])} / "
      f"역수 {len(sw_info['recip'])} / 불변 {len(sw_info['inv'])}")
print('  교환쌍:', [a for a, _ in sw_info['pairs']])
chk = X_ho[:200].copy()
assert np.allclose(swap_X(swap_X(chk)), chk, atol=1e-5), 'involution 실패!'
print('involution 검증 통과 (swap∘swap = 항등)')

In [ ]:
# ── 평가 유틸 — §0 축 분해 항등식 + §7 paired bootstrap ──
EPS = 1e-15

def per_row_ll(p, yy):
    p = np.clip(np.asarray(p, np.float64), EPS, 1.0)
    p = p / p.sum(1, keepdims=True)
    return -np.log(p[np.arange(len(yy)), yy])

def axis_report(p, yy):
    # 총 ll = tie축 + 승부축 (임의의 3-class 확률벡터에 성립하는 항등식, recipe §0)
    from sklearn.metrics import roc_auc_score
    p = np.clip(np.asarray(p, np.float64), EPS, 1.0)
    p = p / p.sum(1, keepdims=True)
    yy = np.asarray(yy)
    total = per_row_ll(p, yy).mean()
    t = np.clip(p[:, 2], EPS, 1 - EPS)
    it = yy == 2
    tie_axis = float(-np.where(it, np.log(t), np.log(1 - t)).mean())
    nz = ~it
    rA = np.clip(p[nz, 0] / (p[nz, 0] + p[nz, 1]), EPS, 1 - EPS)
    win_axis = float(-np.where(yy[nz] == 0, np.log(rA), np.log(1 - rA)).sum() / len(yy))
    assert abs(total - tie_axis - win_axis) < 1e-6, '축 분해 검산 실패'
    return {'총 ll': float(total), 'tie축': tie_axis, '승부축': win_axis,
            'tie AUC': float(roc_auc_score(it.astype(int), t)),
            '승부 AUC': float(roc_auc_score((yy[nz] == 0).astype(int), rA)),
            'acc%': float((p.argmax(1) == yy).mean() * 100),
            'tie recall%': float((p.argmax(1) == 2)[it].mean() * 100)}

def paired_bootstrap(ll_a, ll_b, n_boot=2000, seed=SEED):
    # Δ = mean(ll_a − ll_b) 의 95% CI. 음수면 a 가 좋다. CI 가 0 을 포함하면 노이즈.
    d = np.asarray(ll_a, np.float64) - np.asarray(ll_b, np.float64)
    rng = np.random.default_rng(seed)
    n = len(d)
    means = np.empty(n_boot)
    for i in range(n_boot):
        means[i] = d[rng.integers(0, n, n)].mean()
    return float(d.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

def fmt_ci(mean, lo, hi, thresh=0.0005):
    sig = '유의' if (lo > 0 or hi < 0) else '노이즈'
    star = ' ★' if sig == '유의' and abs(mean) >= thresh else ''
    return f'{mean:+.4f} [{lo:+.4f},{hi:+.4f}] {sig}{star}'

In [ ]:
# ── [정의] 부스터 3종 + 5-fold 학습 (swap 증강 + TTA) ──
import inspect
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

N_LGB, N_XGB, N_CAT = 3000, 4000, 4000          # 스모크/배선 점검 시 줄여서 사용

def make_lgbm():
    return lgb.LGBMClassifier(objective='multiclass', num_class=3, learning_rate=0.03,
                              n_estimators=N_LGB, num_leaves=63, min_child_samples=100,
                              colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                              reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbosity=-1)

def make_xgb():
    return xgb.XGBClassifier(objective='multi:softprob', num_class=3, learning_rate=0.03,
                             n_estimators=N_XGB, max_depth=8, min_child_weight=8,
                             subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                             tree_method='hist', eval_metric='mlogloss',
                             early_stopping_rounds=200, random_state=SEED, n_jobs=-1, verbosity=0)

def make_cat():
    return cb.CatBoostClassifier(loss_function='MultiClass', eval_metric='MultiClass',
                                 iterations=N_CAT, learning_rate=0.03, depth=8,
                                 l2_leaf_reg=4.0, early_stopping_rounds=200,
                                 bootstrap_type='Bernoulli', subsample=0.8, border_count=254,
                                 random_seed=SEED, verbose=0, allow_writing_files=False)

_LGB_NEW = 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters
def fit_one(name, m, Xf, yf, X_es, y_es):
    if name == 'lgbm':
        ekw = {'eval_X': X_es, 'eval_y': y_es} if _LGB_NEW else {'eval_set': [(X_es, y_es)]}
        m.fit(Xf, yf, **ekw, callbacks=[lgb.early_stopping(100, verbose=False)])
    elif name == 'xgb':
        m.fit(Xf, yf, eval_set=[(X_es, y_es)], verbose=False)
    else:
        m.fit(Xf, yf, eval_set=(X_es, y_es), use_best_model=True)

def best_iter(name, m):
    if name == 'lgbm':
        return int(m.booster_.best_iteration or 0)
    if name == 'xgb':
        return int(getattr(m, 'best_iteration', 0) or 0)
    return int(m.get_best_iteration() or 0)

def train_cv(name, factory):
    # 5-fold: 그래디언트 train, ES valid(고정 — twin 격리 자동), 예측은 swap-TTA
    oof = np.zeros((len(X_tr), 3), np.float32)
    p_valid = np.zeros((len(X_va), 3), np.float64)
    p_hold = np.zeros((len(X_ho), 3), np.float64)
    models, iters = [], []
    for k in tqdm(np.sort(np.unique(folds)), desc=name, file=sys.stdout, smoothing=0):
        tr, va = folds != k, folds == k
        Xf = np.vstack([X_tr[tr], swap_X(X_tr[tr])])
        yf = np.hstack([y_tr[tr], swap_label(y_tr[tr])])
        m = factory()
        fit_one(name, m, Xf, yf, X_va, y_va)
        tta = lambda Xp: (m.predict_proba(Xp) + swap_p3(m.predict_proba(swap_X(Xp)))) / 2
        oof[va] = tta(X_tr[va])
        p_valid += tta(X_va) / 5
        p_hold += tta(X_ho) / 5
        models.append(m)
        iters.append(best_iter(name, m))
    return {'oof': oof, 'p_valid': p_valid.astype(np.float32),
            'p_hold': p_hold.astype(np.float32), 'models': models, 'best_iters': iters}
print(f'정의 완료 — lgb {lgb.__version__} / xgb {xgb.__version__} / cat {cb.__version__}')

In [ ]:
# ── 학습 실행 — 3 부스터 ──
R = {}
for name, factory in [('lgbm', make_lgbm), ('xgb', make_xgb), ('cat', make_cat)]:
    t0 = time.time()
    R[name] = train_cv(name, factory)
    from sklearn.metrics import log_loss
    print(f"{name}: best_iters {R[name]['best_iters']} | "
          f"train OOF ll {log_loss(y_tr, R[name]['oof'], labels=[0, 1, 2]):.4f} | "
          f"{(time.time() - t0) / 60:.1f}분")

In [ ]:
# ── 평가 — holdout 전용. 축 분해 + 3 부스터 대조 + logp 평균 앙상블 미리보기 ──
prior = np.bincount(y_tr, minlength=3) / len(y_tr)
logp_mean = np.mean([np.log(np.clip(R[n]['p_hold'], 1e-15, 1)) for n in R], axis=0)
p_ens = np.exp(logp_mean)
p_ens /= p_ens.sum(1, keepdims=True)
preds = {n: R[n]['p_hold'] for n in R}
preds['3부스터 logp 평균'] = p_ens
preds['사전분포(무정보)'] = np.tile(prior, (len(y_ho), 1))
tbl = pd.DataFrame({k: axis_report(p, y_ho) for k, p in preds.items()}).T.sort_values('총 ll')
display(tbl.round(4))

base = per_row_ll(R['lgbm']['p_hold'], y_ho)
for n in ['xgb', 'cat']:
    m_, lo, hi = paired_bootstrap(per_row_ll(R[n]['p_hold'], y_ho), base)
    print(f'{n} − lgbm:', fmt_ci(m_, lo, hi))
m_, lo, hi = paired_bootstrap(per_row_ll(p_ens, y_ho), base)
print('앙상블 − lgbm:', fmt_ci(m_, lo, hi))
print('\n참고: V4 앵커(U-LGBM-swap 1.0066)는 stage2 10,748(valid+hold) 기준이라 직접 비교는 참고만.')
print('holdout 5,748 은 SE ≈ 0.0058 — CI 폭이 stage2 의 1.37배임을 감안해 읽을 것 (recipe §7).')

In [ ]:
# ── 저장 — 모델 3종 + 로짓 npz(outputs/) + 중요도 ──
import lightgbm, xgboost, catboost, sklearn
from datetime import datetime
os.makedirs('model', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

for k, m in enumerate(R['lgbm']['models'], 1):
    m.booster_.save_model(f'model/lgbm_fold{k}.txt')
for k, m in enumerate(R['xgb']['models'], 1):
    m.save_model(f'model/xgb_fold{k}.json')
for k, m in enumerate(R['cat']['models'], 1):
    m.save_model(f'model/cat_fold{k}.cbm')

meta = {
    'version': 2, 'created': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'n_folds': 5, 'use_swap': True, 'cols': UCOLS,
    'flags': {k: bool(v) for k, v in FLAGS.items()},
    'es': {'set': 'origin/valid.csv', 'rounds': {'lgbm': 100, 'xgb': 200, 'cat': 200}},
    'best_iters': {n: R[n]['best_iters'] for n in R},
    'metrics_holdout': {n: axis_report(R[n]['p_hold'], y_ho) for n in R},
    'libs': {'lightgbm': lightgbm.__version__, 'xgboost': xgboost.__version__,
             'catboost': catboost.__version__, 'sklearn': sklearn.__version__},
    'artifacts': ['artifacts/tfidf.joblib', 'artifacts/svd.joblib', 'emb_model/',
                  'feature_catalog.csv'],
}
with open('model/meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

# 로짓(log p) 내보내기 — 2차 부스터/앙상블 재료. 트리 전문가 표현은 logp 확정 (V4 실측: prob 과 동일)
def logp(p):
    return np.log(np.clip(p, 1e-15, 1.0)).astype(np.float32)

np.savez_compressed('outputs/arith_experts.npz',
                    cols=np.array(UCOLS),
                    ids_train=df.loc[m_tr, 'id'].to_numpy(), y_train=y_tr,
                    ids_valid=df.loc[m_va, 'id'].to_numpy(), y_valid=y_va,
                    ids_hold=df.loc[m_ho, 'id'].to_numpy(), y_hold=y_ho,
                    **{f'logp_oof_train__{n}': logp(R[n]['oof']) for n in R},
                    **{f'logp_valid__{n}': logp(R[n]['p_valid']) for n in R},
                    **{f'logp_hold__{n}': logp(R[n]['p_hold']) for n in R})
print('saved: model/{lgbm,xgb,cat}_fold1..5 + model/meta.json')
print('saved: outputs/arith_experts.npz  (train OOF / valid / holdout x 3모델, log p)')

# ── 피쳐별 중요도 — 부스터 3종 각각 + 평균, 전 피쳐 전량 출력 ──
# 중요도 척도는 모델마다 다르다(LGBM/XGB 는 gain = 그 피쳐로 쪼갤 때 줄어든 손실 총합,
# CatBoost 는 PredictionValuesChange = 그 피쳐 값이 바뀔 때 예측이 흔들리는 정도).
# 절대값끼리는 비교가 안 되므로 모델별로 합=100% 로 정규화한 뒤 평균한다.
# 5 fold 평균이고, 상관된 피쳐끼리는 gain 이 임의로 갈리므로 개별 순위는 노이즈가 크다 —
# 판정용이 아니라 '부스터가 무엇을 보고 있나'를 눈으로 확인하는 용도다.
def _imp_lgbm(m):
    return np.asarray(m.booster_.feature_importance('gain'), np.float64)

def _imp_xgb(m):
    sc = m.get_booster().get_score(importance_type='gain')   # 안 쓴 피쳐는 키가 아예 없다
    out = np.zeros(len(UCOLS))
    for k, v in sc.items():
        j = int(k[1:]) if k.startswith('f') and k[1:].isdigit() else UCOLS.index(k)
        out[j] = v
    return out

def _imp_cat(m):
    return np.asarray(m.get_feature_importance(), np.float64)

IMP_FN = {'lgbm': _imp_lgbm, 'xgb': _imp_xgb, 'cat': _imp_cat}
imp = pd.DataFrame({'feature': UCOLS})
for n, fn in IMP_FN.items():
    v = np.mean([fn(m) for m in R[n]['models']], axis=0)     # fold 평균
    imp[f'{n}%'] = v / max(v.sum(), 1e-12) * 100             # 모델별 정규화
imp['평균%'] = imp[[f'{n}%' for n in IMP_FN]].mean(axis=1)
imp['쓴모델'] = (imp[[f'{n}%' for n in IMP_FN]] > 0).sum(axis=1)   # 0 이면 세 부스터 전부 무시
imp = imp.merge(catalog[['feature', '유형', '의존성']], on='feature')
imp = imp.sort_values('평균%', ascending=False).reset_index(drop=True)
imp.insert(0, '순위', np.arange(1, len(imp) + 1))
imp['누적%'] = imp['평균%'].cumsum()
imp.to_csv('model/arith_importance.csv', index=False)
print(f'saved: model/arith_importance.csv — {len(imp)}종 x 부스터 3종\n')

print(f'[전체 피쳐 중요도 — {len(imp)}종 전량, 모델별 % 는 각자 합 100 기준]')
with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                       'display.width', 200):
    display(imp[['순위', 'feature', '유형', 'lgbm%', 'xgb%', 'cat%', '평균%', '누적%', '쓴모델']]
            .round(3))

dead = imp[imp['쓴모델'] == 0]
print(f"\n[한 번도 안 쓰인 피쳐] {len(dead)}종" + (f": {list(dead['feature'])}" if len(dead) else ''))
print(f"[상위 20종이 차지하는 비중] {imp['평균%'].head(20).sum():.1f}%   "
      f"| 상위 50종 {imp['평균%'].head(50).sum():.1f}%")
print('[모델 간 순위 상관(spearman)] '
      + ', '.join(f'{a}-{b} {imp[f"{a}%"].corr(imp[f"{b}%"], method="spearman"):.3f}'
                  for a, b in (('lgbm', 'xgb'), ('lgbm', 'cat'), ('xgb', 'cat'))))

print('\n[유형별 중요도]')
g = imp.groupby('유형').agg(n=('평균%', 'size'), 총합=('평균%', 'sum'),
                           lgbm=('lgbm%', 'sum'), xgb=('xgb%', 'sum'), cat=('cat%', 'sum'))
g['피쳐당'] = g['총합'] / g['n']
display(g.sort_values('총합', ascending=False).round(3))

## 읽는 법 / 다음 단계

1. **3 부스터 대조** — Δ CI 가 노이즈면 세 모델은 같은 정보에 도달한 것. logp 평균 앙상블이
   단독 최고를 유의하게 이기면 표현 다양성이 실재한다는 뜻 — 2차 부스터에 셋 다 넣을 근거.
2. **holdout 만으로 판정** — valid 는 ES 로 오염됐으므로 성능 주장에 쓰지 않는다.
   `outputs/arith_experts.npz` 의 valid 로짓은 2차 앙상블 학습 재료로만 쓴다.
3. **배포 번들** = `Features.ipynb` 의 추출 정의 + `feature_catalog.csv` + `artifacts/` +
   `emb_model/` + `model/`. 새 데이터 추론 = 같은 정의로 피쳐 재계산(transform 모드) →
   카탈로그 순서로 정렬 → fold 모델 TTA 평균.
4. **다음**: [Ablation.ipynb](Ablation.ipynb) 로 유형 플래그 제거실험. 채택 변경이 나오면
   이 노트북의 FLAGS 를 바꿔 재학습하고 meta.json 을 갱신한다.